# day-17-chunking-strategies — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [10]:
# ---- Solution 1 ----
os_ = [0, 25, 50, 100, 150]
recs = [evaluate(fixed(DOC, 250, o))["recall_at_k"] for o in os_]
ns   = [len(fixed(DOC, 250, o)) for o in os_]
for o, r, n in zip(os_, recs, ns):
    print(f"overlap={o:3d}  recall@3={r:.2f}  chunks={n}")
print("marginal recall per extra chunk from 0->150 overlap:",
      round((recs[-1]-recs[0]) / max(1, ns[-1]-ns[0]), 3))

overlap=  0  recall@3=0.86  chunks=12
overlap= 25  recall@3=0.95  chunks=14
overlap= 50  recall@3=0.95  chunks=15
overlap=100  recall@3=1.00  chunks=20
overlap=150  recall@3=1.00  chunks=30
marginal recall per extra chunk from 0->150 overlap: 0.008


In [11]:
# ---- Solution 2 ----
strategies = {
 "fixed 250 ov20": fixed(DOC, 250, 50),
 "sentence 400": by_sentence(DOC, 400),
 "paragraph": by_paragraph(DOC),
 "recursive 400": recursive(DOC, 400),
 "semantic 25%": semantic(DOC, 25),
}
print(f"{'strategy':18s} {'r@1':>5} {'r@3':>5} {'r@5':>5} {'#chunks':>8}")
for name, ch in strategies.items():
    r1, r3, r5 = (evaluate(ch, k)["recall_at_k"] for k in (1, 3, 5))
    print(f"{name:18s} {r1:>5.2f} {r3:>5.2f} {r5:>5.2f} {len(ch):>8}")

strategy             r@1   r@3   r@5  #chunks


fixed 250 ov20      0.95  0.95  1.00       15


sentence 400        0.90  1.00  1.00        8


paragraph           1.00  1.00  1.00        7


recursive 400       1.00  1.00  1.00       12


semantic 25%        1.00  1.00  1.00       10


In [12]:
# ---- Solution 6 ----
import tiktoken
tk = tiktoken.get_encoding("cl100k_base")
qv = encode([q for q, _ in QA])
sent_chunks = by_sentence(DOC, 300); cv = encode(sent_chunks)
a_tokens = np.mean([sum(len(tk.encode(sent_chunks[i]))
                        for i in np.argsort(-(cv @ qv[j]))[:3]) for j in range(len(QA))])
b_tokens = np.mean([sum(len(tk.encode(p)) for p in small_to_big(q, k=2)) for q, _ in QA])
print(f"S6: top-3 sentence chunks  ~{a_tokens:.0f} tokens/query")
print(f"    small-to-big top-2 parents ~{b_tokens:.0f} tokens/query")
print("    trade: small-to-big spends more prompt tokens to guarantee the retrieved fact has")
print("    its full surrounding context (fewer 'retrieved but unanswerable' cases).")

S6: top-3 sentence chunks  ~150 tokens/query
    small-to-big top-2 parents ~141 tokens/query
    trade: small-to-big spends more prompt tokens to guarantee the retrieved fact has
    its full surrounding context (fewer 'retrieved but unanswerable' cases).


### Solutions 3, 4, 5 (sketch)

**S3:** at `fixed(DOC, ~130)` the Security section's "Multi-factor authentication is mandatory
for all systems." can land such that "mandatory for all systems" starts a new chunk. The query
*"is 2FA required"* then matches neither half well. `by_sentence` keeps the sentence whole;
overlap ≥ 20% also covers it.

**S4:** low `pct` (5–10) → almost no splits → one huge chunk, recall collapses on specific
questions (blur). High `pct` (50–60) → splits at every mild topic wobble → sentence-sized
chunks, recall drops because facts lose context and embed noisily. Best around 20–30% on this
doc; mean chunk length ~150–350 chars there.

**S5:** the heading helps **most at small sizes** (150): a 150-char chunk from the Expenses
section might be just "Meal per diem is $60 domestic and $90 international." — prepending
`[Expenses]` adds the topical anchor the short text lacks. At 600 chars the chunk already
contains enough context that the heading adds little.

### Answer key
1. Retrievability (chunk embedding matches the right queries — hurt by chunks too big or facts
   split) vs answerability (chunk has enough context to actually answer — hurt by chunks too
   small).
2. Its embedding is the average of several topics, so it matches many queries weakly and few
   queries strongly — it gets retrieved for the wrong questions and ranked low for the right
   one (MRR drops).
3. It keeps a fact that straddles a chunk boundary intact in at least one chunk. Cost: more
   chunks to store/index and redundant text repeated across chunks (and in the prompt).
4. When the document lacks reliable structure markers — transcripts, scraped HTML, OCR — so
   heading/paragraph splitting produces bad boundaries. On clean structured docs, recursive
   usually matches it for far less compute.
5. Retrieve against small, precise chunks (good match quality), but pass the larger parent
   section to the LLM. Optimises retrieval precision and answer context simultaneously, at the
   cost of more prompt tokens.
6. recall@k on a QA set: did some retrieved chunk actually contain the gold fact? Almost every
   "RAG got worse" is a retrieval-recall regression, and chunking is the usual cause.
7. A short chunk often lacks topical signal; the heading adds it (improving the embedding
   match) and hands the LLM the section name for free (better, more citable answers).